# X-Mobility 파인튜닝 — Kaggle (E · 서진석)

`e2e/finetune.py` 를 Kaggle 무료 GPU(T4 × 2)에서 돌린다. 데이터 = 교사 데이터 xm_v1(MANIFEST `xm_parking_r3` v1.0), 시작 가중치 = `nvidia/X-Mobility` 사전학습 ckpt.

**한 번만 준비**
1. Kaggle 계정 휴대폰 인증(인증해야 GPU · 인터넷 켜짐).
2. **데이터셋 만들기**: Datasets → New Dataset → `02_xm_v1.zip` 업로드(이름 예: `fd-xm-v1`, Private). Kaggle 이 zip 을 풀어 둠.
3. **HF 토큰**: huggingface.co 에서 `nvidia/X-Mobility` 약관 동의 → Settings → Access Tokens(Read) → 이 노트북 Add-ons → Secrets 에 이름 `HF_TOKEN` 으로 저장, 노트북에 연결(체크).
4. 이 노트북 오른쪽 Settings: **Accelerator = GPU T4 x2 · Internet = On**, Input 에 위 데이터셋 추가.

**돌리기**
- 처음: 아래 `MODE = "smoke"` 로 Run All(약 15~25분) — 실행기 · 데이터 · ckpt 확인 + 스텝 시간 실측.
- 그다음: `MODE = "full"` 로 바꾸고 **Save Version → Save & Run All (Commit)** — 브라우저를 닫아도 최대 12시간 돈다. 시간 예산에 맞춰 에폭 수를 자동으로 정함.
- 결과: 버전의 Output 탭에서 `run_meta.json` · `checkpoints/` 다운로드 → MANIFEST `checkpoint` 행.

In [ ]:
MODE = "smoke"            # "smoke" → 40스텝 시험(1에폭 = 40배치) · "full" → 본 학습
TIME_BUDGET_H = 11.0      # full: 학습에 쓸 시간(Kaggle 세션 12시간 - 준비 · test 여유)
MAX_EPOCHS = 10           # full: 시간이 남아도 이 이상은 안 돌림(gin 기본값)
REPO_URL = "https://github.com/JP-Avengers/FUTURE-DRIVE-PROJECT.git"
XM_URL, XM_REV = "https://github.com/NVlabs/X-Mobility.git", "5837aea"
CKPT_REPO, CKPT_FILE = "nvidia/X-Mobility", "x_mobility-nav2-semantic_action_path.ckpt"
WORK = "/kaggle/working"

In [ ]:
import os, subprocess, glob, json, math, shutil, time
def sh(cmd, check=True):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    if check and r.returncode != 0:
        raise SystemExit(f"실패({r.returncode}): {cmd}")
    return r.returncode

sh("nvidia-smi --query-gpu=name,memory.total --format=csv")
os.chdir("/kaggle/tmp" if os.path.isdir("/kaggle/tmp") else "/tmp")
if not os.path.isdir("FUTURE-DRIVE-PROJECT"):
    sh(f"git clone --depth 20 {REPO_URL}")
if not os.path.isdir("X-Mobility"):
    sh(f"git clone {XM_URL} && cd X-Mobility && git checkout -q {XM_REV}")
REPO, XM = os.path.abspath("FUTURE-DRIVE-PROJECT"), os.path.abspath("X-Mobility")
sh(f"git -C {REPO} log --oneline -1 && git -C {XM} log --oneline -1")
assert "--max-time" in open(f"{REPO}/e2e/finetune.py").read(), "레포 main 에 finetune.py 새 판(--max-time)이 아직 없음 — push 먼저"

In [ ]:
# X-Mobility 가 맞춘 판(requirements.txt)과 같게. torch 는 Kaggle 기본(CUDA) 그대로.
# huggingface_hub 0.26+ 는 diffusers 0.29 가 쓰는 cached_download 를 지워서 0.25 로 고정.
sh("pip install -q gin-config==0.5.0 einops==0.7.0 pytorch-lightning==2.5.0.post0 transformers==4.48.1 "
   "diffusers==0.29.2 huggingface_hub==0.25.2 timm==1.0.14 torcheval==0.0.7 torchmetrics")
sh("python -c \"import torch, transformers, diffusers, pytorch_lightning as pl; "
   "print(torch.__version__, torch.cuda.device_count(), transformers.__version__, diffusers.__version__, pl.__version__)\"")

In [ ]:
# 데이터 루트 = train/val/test 가 있는 폴더
cand = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/train", recursive=True)
        if os.path.isdir(os.path.join(os.path.dirname(p), "val")) and os.path.isdir(os.path.join(os.path.dirname(p), "test"))]
assert cand, "입력 데이터셋에서 train/val/test 를 못 찾음 — 오른쪽 Input 에 xm_v1 데이터셋 추가"
DATA = sorted(cand, key=len)[0]
print("DATA =", DATA)
sh(f"python {REPO}/e2e/check_xm_dataset.py {DATA} --image-stride 20")

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download
token = UserSecretsClient().get_secret("HF_TOKEN")
CKPT = hf_hub_download(CKPT_REPO, CKPT_FILE, token=token, local_dir=os.path.abspath("ckpt"))
print(CKPT, round(os.path.getsize(CKPT) / 2**30, 2), "GB")

In [ ]:
BASE = (f"python {REPO}/e2e/finetune.py --xm-repo {XM} -c {REPO}/e2e/configs/finetune_parking_r3.gin "
        f"-d {DATA} -p {CKPT} --wandb offline --save-top-k 1")
smoke = f"{WORK}/smoke"
if not os.path.exists(f"{smoke}/run_meta.json") or "step_time_s" not in json.load(open(f"{smoke}/run_meta.json")):
    shutil.rmtree(smoke, ignore_errors=True)
    t0 = time.time()
    sh(f"{BASE} -o {smoke} --limit-train-batches 40 --limit-val-batches 4 --no-test --gin NUM_EPOCHS=1")
    print(f"시험 전체 {time.time() - t0:.0f} s")
m = json.load(open(f"{smoke}/run_meta.json"))
k = m["keys"] or {}
print(json.dumps({x: m.get(x) for x in ("step_time_s", "steps_per_epoch", "world_size", "peak_gpu_mem_gb", "global_step")}, indent=1))
print("missing", len(k.get("missing", [])), "· 예상 밖 unexpected", len(k.get("unexpected_other", [])), "· ckpt hparams", k.get("ckpt_hparams"))
assert not k.get("missing"), "missing 키 있음 — 멈춤"

In [ ]:
# full: 실측 스텝 시간으로 에폭 수를 정함(검증 · 저장 여유 15 %). lr 스케줄(OneCycle)이 이 에폭 수에 맞춰 끝까지 내려감.
if MODE == "full":
    per_epoch_s = m["steps_per_epoch"] * m["step_time_s"] * 1.15
    epochs = max(1, min(MAX_EPOCHS, int((TIME_BUDGET_H * 3600 - 1200) // per_epoch_s)))
    print(f"에폭당 약 {per_epoch_s / 60:.0f}분 → {epochs} 에폭")
    out = f"{WORK}/xm_ft_kaggle_e{epochs}"
    hh = int(TIME_BUDGET_H); mm = int((TIME_BUDGET_H - hh) * 60)
    sh(f'{BASE} -o {out} --gin "NUM_EPOCHS={epochs}" --max-time "00:{hh:02d}:{mm:02d}:00"', check=False)
    print(open(f"{out}/run_meta.json").read())
else:
    print("smoke 끝 — 위 숫자를 확인하고 MODE='full' 로 Save & Run All(Commit)")